# GAE_HEP on Kaggle GPU: hyper-parameter study of the energy-flow graph VAE

Permutation-invariant graph autoencoders for unsupervised anomaly detection on jets of the LHC Olympics 2020 R&D dataset ([github.com/raktim711/GAE_HEP](https://github.com/raktim711/GAE_HEP); results and discussion in `RESULTS.md`).

This notebook runs the GPU part of the hyper-parameter study (`gaehep sweep`):

1. data: download (Zenodo record 6466204, about 3.2 GB), clustering, tables, splits, single-observable baselines;
2. **grid sweep** under the development protocol (validation splits only): latent size x VAE beta x encoder width x training budget (72 points, random order, one seed each);
3. **seed confirmation** of the 3 best grid points (3 seeds), then the best configuration with **5 seeds scored on the test splits** (2-prong and the never-tuned 3-prong signal) and a seed ensemble;
4. mass decorrelation, bump hunt and the weak-supervision study for the best configuration, plus scale variants (100 constituents, deeper encoder, more neighbours) of the best grid point;
5. reports and a zip of all results (`/kaggle/working/gaehep_results.zip`, refreshed after every phase, so a time-out still leaves the finished part).

> **Before you run:** in the Kaggle session options (right-hand panel) switch **Internet on** (needed for `git clone`, `pip install` and the Zenodo download) and select a **GPU accelerator**. Run it with *Save Version -> Save & Run All (Commit)*: the whole study takes about 8-9 h.

In [ ]:
!nvidia-smi

import os

REPO_URL = "https://github.com/raktim711/GAE_HEP.git"
REPO_DIR = "/kaggle/working/GAE_HEP"

# clone once; skipped when the repo is already there (e.g. after a kernel restart)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .

import torch

print("torch", torch.__version__, "| built for CUDA", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU visible - choose a GPU accelerator in the session options.")

In [ ]:
import os

# Kaggle: keep everything under /kaggle/working
os.environ["GAEHEP_DATA"] = "/kaggle/working/data"
os.environ["GAEHEP_RESULTS"] = "/kaggle/working/results"

# data pipeline; the shell commands inherit os.environ.
# `download` is md5-verified and resumable: re-run this cell if it gets interrupted.
!gaehep download
!gaehep preprocess --sample rnd --workers 4
!gaehep preprocess --sample qqq --workers 4
!gaehep tables --sample rnd
!gaehep tables --sample qqq
!gaehep splits
!gaehep baselines

## GPU study

Everything goes through the `gaehep` command line; each step is idempotent, so a re-run reuses finished runs. The sweep has a wall-clock budget (`--budget-hours`): it stops launching grid points after 60 % of it and seed runs after 85-90 %, so the notebook ends within Kaggle's 12 h limit.

In [ ]:
import json
import os
import shlex
import subprocess
import time
import zipfile
from pathlib import Path

REPO_DIR = "/kaggle/working/GAE_HEP"
RESULTS = Path("/kaggle/working/results")
ZIP_PATH = Path("/kaggle/working/gaehep_results.zip")

# set again so that this cell (and the next ones) also work after a kernel restart
os.environ["GAEHEP_DATA"] = "/kaggle/working/data"
os.environ["GAEHEP_RESULTS"] = str(RESULTS)
os.chdir(REPO_DIR)
T_START = time.time()


def gaehep(*args, check=True):
    """Run `gaehep <args>`; the log streams into the cell output."""
    cmd = ["gaehep", *map(str, args)]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, check=check, cwd=REPO_DIR, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    print(f"  ({(time.time() - t0) / 60:.1f} min, notebook at {(time.time() - T_START) / 3600:.2f} h)", flush=True)
    return r.returncode


def zip_results():
    """Zip the results (no model weights / checkpoints / large score arrays) for download."""
    n = 0
    with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for f in RESULTS.rglob("*"):
            if f.is_file() and f.suffix not in {".pt", ".ckpt"} and "checkpoint" not in f.name and f.stat().st_size < 50e6:
                z.write(f, f.relative_to(RESULTS.parent))
                n += 1
    print(f"zipped {n} files -> {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)", flush=True)


GPU = ["--set", "train.device=cuda", "--set", "train.threads=4"]

In [ ]:
# ---- phase 1-3: grid sweep -> seed confirmation -> best configuration on the test splits (5 seeds) ----
gaehep("sweep", "configs/ef_vae.yaml", "--prefix", "gpu", "--device", "cuda", "--threads", "4",
       "--budget-hours", "7", "--top-k", "3", "--seeds", "3", "--final-seeds", "5")
zip_results()

# ---- phase 4: post-hoc analyses of the best configuration ----
gaehep("qn", "gpu_best_seeds", check=False)
gaehep("bumphunt", "gpu_best_seeds", "--agg", "prod", check=False)
gaehep("weak", "gpu_best_s1", "--n-models", "5", check=False)
zip_results()

# ---- phase 5: scale variants of the best grid point (dev protocol) ----
best = json.loads((RESULTS / "dev" / "gpu_sweep.json").read_text())["best"]["point"]
point = ["--set", f"model_args.latent_dim={best['latent_dim']}", "--set", f"model_args.beta={best['beta']}",
         "--set", f"model_args.variational={'true' if best['beta'] > 0 else 'false'}",
         "--set", f"train.jets_per_epoch={best['jets_per_epoch']}",
         "--set", "extra_scores=[kl]" if best["beta"] > 0 else "extra_scores=[]"]
w = best["width"]
SCALE = {
    "n100": [f"model_args.hidden=[{w},{w},{w}]", "data.n_const=100", "model_args.k=16"],
    "deep": [f"model_args.hidden=[{w},{w},{w},{w}]"],
    "k24": [f"model_args.hidden=[{w},{w},{w}]", "model_args.k=24"],
    "nout64": [f"model_args.hidden=[{w},{w},{w}]", "model_args.n_out=64"],
}
for tag, over in SCALE.items():
    if (time.time() - T_START) / 3600 > 10.5:
        break
    extra = [x for o in over for x in ("--set", o)]
    gaehep("dev", "configs/ef_vae.yaml", "--name", f"gpu_scale_{tag}", *point, *extra, *GPU, check=False)

gaehep("report", check=False)
gaehep("dev-report", check=False)
zip_results()

In [ ]:
import os
import zipfile
from pathlib import Path

from IPython.display import FileLink, Markdown, display

RESULTS = Path(os.environ.get("GAEHEP_RESULTS", "/kaggle/working/results"))
ZIP_PATH = Path("/kaggle/working/gaehep_results.zip")

# 1) the summary table written by `gaehep report`
summary = RESULTS / "summary.md"
if summary.is_file():
    display(Markdown(summary.read_text()))
else:
    print(f"{summary} not found - run the previous cell (it ends with `gaehep report`) first.")


# 2) zip the results for download, leaving out model weights / checkpoints
def is_checkpoint(path: Path) -> bool:
    parts = [p.lower() for p in path.relative_to(RESULTS).parts]
    return path.suffix.lower() in {".pt", ".pth", ".ckpt"} or any(
        "checkpoint" in p or p.startswith("ckpt") for p in parts
    )


added = skipped = 0
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(RESULTS.rglob("*")):
        if not path.is_file():
            continue
        if is_checkpoint(path):
            skipped += 1
            continue
        zf.write(path, arcname=str(Path("results") / path.relative_to(RESULTS)))
        added += 1

print(f"{ZIP_PATH}: {added} files, {ZIP_PATH.stat().st_size / 1e6:.1f} MB "
      f"({skipped} *.pt / checkpoint files left out)")

# download link (the file also shows up in the notebook's Output tab)
cwd = os.getcwd()
try:
    os.chdir(ZIP_PATH.parent)
    display(FileLink(ZIP_PATH.name))
finally:
    os.chdir(cwd)